# Parallel step 50b — merge and verify saved source catalogs

Run after 50a. This reads the individual QuakeML files back from disk, checks their hashes and merges by their shared event identity. It preserves every origin and magnitude rather than selecting one solution per source. Observation-only containers and actual picks from step 32 are retained.

Preferred origins follow the existing step-43 selection. Preferred magnitudes use the explicit priority below: BRUN first, then the existing HYPO71 preference, then GOLD and PEPI as additional fallbacks. No magnitude values are recalibrated; preferred magnitudes across events remain on different source scales. Within a source, ties use origin time then source line, not inferred solution quality.

This is a parallel QuakeML product. Do not feed it to the existing Nordic exporter until that export has been reviewed; this trial does not replace production step 50 or step 80.

In candidate-aware mode, the observation intermediate also contains unresolved pick containers and candidates not linked to historical origins. These are not confirmed earthquake counts. All input picks must survive, and every historical Origin and Magnitude remains exactly once.


In [ ]:
from pathlib import Path
import os, sys
from IPython.display import display

# Works from this subdirectory, 10_conversion, or the repository root.
roots = [Path.cwd(), *Path.cwd().parents]
if os.environ.get('PINATUBO_CODE_ROOT'):
    roots.insert(0, Path(os.environ['PINATUBO_CODE_ROOT']))
candidates = [p for r in roots for p in (r, r / '10_conversion')]
PIPELINE_DIR = next(p for p in candidates if (p / 'lib' / 'clock.py').is_file())
CODE_DIR = next(p for p in [Path.cwd(), PIPELINE_DIR / 'catalog_parallel']
                if (p / 'catalog_workflow.py').is_file())
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(CODE_DIR))
from lib.config import pipeline_paths
from catalog_workflow import build_sources, merge_sources
P = pipeline_paths()

# Both notebooks must use the same trial root. Choose a NEW name for a fresh run.
RUN_NAME = 'candidate_trial_01'
TRIAL_ROOT = Path(os.environ.get('PINATUBO_CATALOG_TRIAL_ROOT',
                 str(Path(P['OUTPUT_ROOT']) / 'catalog_parallel' / RUN_NAME)))
SOURCE_DIR = TRIAL_ROOT / 'sources'
MERGED_DIR = TRIAL_ROOT / 'merged'
print('Existing inputs:', P['OUTPUT_ROOT'])
print('Individual catalogs:', SOURCE_DIR)
print('Merged catalog:', MERGED_DIR)

CANDIDATE_DIR = TRIAL_ROOT / 'candidates'


In [ ]:
MAGNITUDE_PRIORITY = ('brun', 'pinaall', 'hypo40', 'gold', 'pepi')
catalog, report = merge_sources(SOURCE_DIR, MERGED_DIR, magnitude_priority=MAGNITUDE_PRIORITY)
display(report)

## Inspect the combined catalog and provenance

The verification checks counts, unique IDs, magnitude-to-origin references, preferred references, source membership and provenance after writing and reading the merged QuakeML. It verifies serialization and accounting; it does not prove the scientific associations or calibrate magnitudes.

In [ ]:
from obspy import read_events
from catalog_workflow import get
combined = read_events(str(MERGED_DIR / 'catalog_combined.xml'))
pinatubo_all = read_events(str(SOURCE_DIR / 'hypo40.xml'))
print('Combined containers:', len(combined))
print('Pinatubo_all events:', len(pinatubo_all))
example = next(e for e in combined if len(e.origins) > 1 and e.magnitudes)
display(get(example))
display([dict(source=get(m)['source'], magnitude=m.mag, origin_id=str(m.origin_id)) for m in example.magnitudes])